# VocEd Lab 2 — Thresholding, Denoising, and Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/Labs/Lab2_thresholding_denoising_evaluation.ipynb)

In Lab 1 we found that background, cytoplasm and nucleus pixels have different typical
brightness.  Now we turn that observation into our first **segmenter** — a rule that labels
every pixel — and then ask the questions every scientist should ask of a result:

- *How good is it?*  We need a number, and the right one.
- *Can we make it better?*  We try cleaning up the image (denoising) before we segment.
- *Would it work on new images?*  We test it honestly, on images it never saw while we were tuning it.

By the end of this lab you will be able to:
- Build a two-threshold segmenter using `True`/`False` masks.
- Compute accuracy, precision, recall, the **Dice score** and **IoU**, and explain why accuracy can mislead.
- Add noise to an image, remove it with Gaussian and median filters, and measure the effect.
- Split data into training and test sets, tune on one and report on the other.

**Textbook:** [Chapter 2](https://obooks.tech/imageprocessing/chapter2.html) (thresholding, noise and denoising) and
[Appendix C](https://obooks.tech/imageprocessing/appendixc.html) (training, testing and evaluation).

---

### How the puzzles work 🧩

Replace every `___` with the right code and run the cell — a ✅ means you got it.  Every
puzzle links to the textbook section that explains it and has hints you can click open.
Running a cell before filling it in gives `NameError: name '___' is not defined`; that is expected.

## 0. Setup — Get the Data

This repeats what we did in Lab 1: copy the repository, load the images and masks, and
convert every image to grayscale with the BT.601 formula.

In [ ]:
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.stack([np.load(f"imagedata/X/{i}.npy") for i in range(200)])   # (200, 3, 256, 256)
y = np.stack([np.load(f"imagedata/y/{i}.npy") for i in range(200)])   # (200, 256, 256)

gray_all = 0.299 * X[:, 0] + 0.587 * X[:, 1] + 0.114 * X[:, 2]        # (200, 256, 256)

gray7 = gray_all[7]
mask7 = y[7]
print("Loaded", X.shape[0], "images")

## 1. Thresholding

**Thresholding** means comparing every pixel with a number.  The answer is a `True`/`False`
array of the same shape as the image, called a **boolean mask**.

📖 Book: [§2.5 Thresholding](https://obooks.tech/imageprocessing/chapter2.html#thresholding) and
[§2.5.1 Boolean Operations on Arrays](https://obooks.tech/imageprocessing/chapter2.html#boolean-operations-on-arrays)

In [ ]:
dark = gray7 < 0.45

print("dark has shape", dark.shape, "and type", dark.dtype)
print("Pixels darker than 0.45:", dark.sum())

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(gray7, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Image 7, grayscale')
axes[1].imshow(dark, cmap='gray')
axes[1].set_title('gray7 < 0.45   (white = True)')
for ax in axes:
    ax.axis('off')
plt.show()

One threshold splits pixels into two groups.  We have **three** classes, so we need **two** thresholds:

| Condition | Label |
|---|---|
| `gray < t_nucleus` | 2 (nucleus) |
| `t_nucleus <= gray < t_background` | 1 (cytoplasm) |
| `gray >= t_background` | 0 (background) |

### 🧩 Puzzle 1 — A two-threshold segmenter

Start from an all-background prediction (all zeros), then overwrite the nucleus and
cytoplasm pixels.  Use `&` ("and") to combine two conditions — each condition needs its own
brackets.

📖 Book: [§2.5.1 Boolean Operations on Arrays](https://obooks.tech/imageprocessing/chapter2.html#boolean-operations-on-arrays) and
[§2.5.2 Visualizing Masks](https://obooks.tech/imageprocessing/chapter2.html#visualizing-masks)

<details><summary><b>💡 Hint 1</b> (click to open)</summary>

`pred7[condition] = value` writes `value` into every pixel where `condition` is `True`.

</details>

<details><summary><b>💡 Hint 2</b> (click to open)</summary>

The cytoplasm band sits *between* the two thresholds: at least `t_nucleus` and below `t_background`.

</details>

After it works, **replace 0.45 and 0.85 with the numbers you wrote down at the end of Lab 1**
and run again.  Which looks better?

In [ ]:
t_nucleus    = 0.45   # darker than this      → nucleus
t_background = 0.85   # at least this bright  → background

pred7 = np.zeros((256, 256), dtype=np.int64)                    # start: everything background (0)
pred7[gray7 < t_nucleus] = ___
pred7[(gray7 >= t_nucleus) & (gray7 < ___)] = ___

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(X[7].transpose(1, 2, 0))
axes[0].set_title('Image 7')
axes[1].imshow(mask7, cmap='viridis', vmin=0, vmax=2, interpolation='nearest')
axes[1].set_title('Ground truth')
axes[2].imshow(pred7, cmap='viridis', vmin=0, vmax=2, interpolation='nearest')
axes[2].set_title(f'Prediction  (t_nucleus={t_nucleus}, t_background={t_background})')
for ax in axes:
    ax.axis('off')
plt.show()

# ── Check ──
assert (pred7 == 2).sum() == (gray7 < t_nucleus).sum(), "Nucleus pixels should get label 2"
assert (pred7 == 0).sum() == (gray7 >= t_background).sum(), "Only pixels at least t_background bright should stay 0"
print("✅ Puzzle 1 solved!")

## 2. How Good Is It?  Measuring Segmentation Quality

Looking at pictures gives an impression.  To compare two segmenters — or to prove to someone
else that ours works — we need **numbers**.

### 2.1 Accuracy, and why it can lie

**Accuracy** is the fraction of pixels we labelled correctly.

📖 Book: [Appendix C — General Machine Learning Metrics](https://obooks.tech/imageprocessing/appendixc.html#general-machine-learning-metrics)

### 🧩 Puzzle 2 — Pixel accuracy

`pred7 == mask7` is `True` wherever the prediction is right.  Turn it into a fraction.

<details><summary><b>💡 Hint</b> (click to open)</summary>

`True` counts as 1 and `False` as 0.  Which NumPy method turns a list of 1s and 0s into the fraction of 1s?

</details>

In [ ]:
accuracy = (pred7 == mask7).___()
print(f"Pixel accuracy on image 7: {accuracy:.3f}")

# ── Check ──
assert abs(accuracy - np.sum(pred7 == mask7) / 65536) < 1e-9, "Accuracy is the number correct divided by the number of pixels"
print("✅ Puzzle 2 solved!")

Now a warning.  Suppose all we care about is **finding nuclei**, and someone hands us a
"segmenter" that simply says *no pixel is ever nucleus*.  It finds **zero** nuclei.
What is its accuracy at the question "nucleus or not?" over the whole dataset?

In [ ]:
lazy_accuracy = (y != 2).mean()
print(f"Accuracy of 'there is never any nucleus': {lazy_accuracy:.3f}")

About 80% — because 80% of pixels really are *not* nucleus.  A useless segmenter looks
good.  This happens whenever one class is much rarer than the others (**class imbalance**).
Always compare your score against a lazy baseline like this one before you believe it.

### 2.2 Precision and recall

To judge the nucleus on its own, sort every pixel into one of four boxes:

|  | truly nucleus | truly not nucleus |
|---|---|---|
| **predicted nucleus** | true positive (TP) | false positive (FP) |
| **predicted not nucleus** | false negative (FN) | true negative (TN) |

- **Precision** = TP / (TP + FP): *of the pixels we called nucleus, how many really are?*
- **Recall** = TP / (TP + FN): *of the real nucleus pixels, how many did we find?*

### 🧩 Puzzle 3 — Count TP, FP and FN

`~` flips a boolean mask (`True` ↔ `False`), so `~true_nuc` means "truly **not** nucleus".

📖 Book: [Appendix C — General Machine Learning Metrics](https://obooks.tech/imageprocessing/appendixc.html#general-machine-learning-metrics)

<details><summary><b>💡 Hint 1</b> (click to open)</summary>

A false positive is a pixel we *predicted* nucleus that is *truly not* nucleus.

</details>

<details><summary><b>💡 Hint 2</b> (click to open)</summary>

A false negative is a pixel that is *truly* nucleus that we did *not predict* as nucleus.

</details>

In [ ]:
pred_nuc = (pred7 == 2)
true_nuc = (mask7 == 2)

TP = (pred_nuc & true_nuc).sum()
FP = (pred_nuc & ___).sum()
FN = (___ & true_nuc).sum()

precision = TP / (TP + ___)
recall    = TP / (TP + ___)

print(f"TP = {TP},  FP = {FP},  FN = {FN}")
print(f"Nucleus precision: {precision:.3f}")
print(f"Nucleus recall:    {recall:.3f}")

# ── Check ──
assert TP + FN == true_nuc.sum(), "TP + FN should equal the number of true nucleus pixels"
assert TP + FP == pred_nuc.sum(), "TP + FP should equal the number of predicted nucleus pixels"
print("✅ Puzzle 3 solved!")

**Think about it:** is our segmenter missing nucleus (low recall) or over-calling it
(low precision) on image 7?  Which threshold would you move, and in which direction, to fix it?

### 2.3 The Dice score

Precision and recall pull in opposite directions, so we usually combine them.  In
segmentation the standard single number is the **Dice score**:

$$\text{Dice}(A, B) = \frac{2\,|A \cap B|}{|A| + |B|}$$

where $A$ is the set of predicted pixels for a class, $B$ the true pixels, and $|A \cap B|$
the pixels in both.  Dice is **1** for perfect overlap and **0** for none.

📖 Book: [Appendix C — Dice Coefficient](https://obooks.tech/imageprocessing/appendixc.html#sec-dice)

### 🧩 Puzzle 4 — Write `dice_score`

<details><summary><b>💡 Hint 1</b> (click to open)</summary>

`A ∩ B` (“A and B”) is `pred_mask & target_mask`.

</details>

<details><summary><b>💡 Hint 2</b> (click to open)</summary>

Look at the formula: the top line has a 2 in it.

</details>

In [ ]:
def dice_score(pred, target, label):
    pred_mask   = (pred   == label)
    target_mask = (target == label)
    intersection = (pred_mask & target_mask).sum()
    denominator  = pred_mask.sum() + target_mask.sum()
    if denominator == 0:
        return 1.0          # the class is absent from both → perfect agreement
    return ___ / denominator

dice_nuc = dice_score(pred7, mask7, 2)
f1_nuc   = 2 * precision * recall / (precision + recall)

print(f"Nucleus Dice on image 7:            {dice_nuc:.4f}")
print(f"Nucleus F1 from precision & recall: {f1_nuc:.4f}")

# ── Check ──
assert abs(dice_score(mask7, mask7, 2) - 1) < 1e-9, "A mask compared with itself should score 1"
assert abs(dice_nuc - f1_nuc) < 1e-9, "Compare your code with the formula"
print("✅ Puzzle 4 solved!")

The two numbers match: **Dice is the same thing as the F1 score** (the average of precision
and recall used in Appendix C).  Remember the images from Lab 1 with no nucleus at all?
The `if denominator == 0` line handles them: if neither the prediction nor the truth has any
nucleus, the segmenter was right, so it scores 1.

### 2.4 Intersection over Union (IoU)

A close cousin of Dice, popular in computer-vision competitions:

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

$|A \cup B|$ ("A or B") counts pixels that are in *either* set.

### 🧩 Puzzle 5 — Write `iou_score`

📖 Book: [Appendix C — Intersection over Union](https://obooks.tech/imageprocessing/appendixc.html#sec-iou)

<details><summary><b>💡 Hint</b> (click to open)</summary>

`&` means *and*.  The symbol for *or* on boolean arrays is the vertical bar.

</details>

In [ ]:
def iou_score(pred, target, label):
    pred_mask   = (pred   == label)
    target_mask = (target == label)
    intersection = (pred_mask & target_mask).sum()
    union        = (pred_mask ___ target_mask).sum()
    if union == 0:
        return 1.0
    return intersection / union

iou_nuc = iou_score(pred7, mask7, 2)
print(f"Nucleus IoU on image 7:  {iou_nuc:.4f}")
print(f"2·IoU / (1 + IoU):       {2 * iou_nuc / (1 + iou_nuc):.4f}   ← compare with the Dice above")

# ── Check ──
assert abs(2 * iou_nuc / (1 + iou_nuc) - dice_nuc) < 1e-9, "Union means pixels in either mask"
print("✅ Puzzle 5 solved!")

IoU is always the stricter (lower) of the two, and one can be converted into the other.  In
this course we report **Dice**.

### 2.5 Per-class scores, and why we leave background out

📖 Book: [Appendix C — Per-Class Metrics](https://obooks.tech/imageprocessing/appendixc.html#sec-per-class-metrics)

In [ ]:
names = ['background', 'cytoplasm', 'nucleus']

print(f"{'class':>12}   Dice    IoU")
for label in [0, 1, 2]:
    print(f"{names[label]:>12}  {dice_score(pred7, mask7, label):.3f}  {iou_score(pred7, mask7, label):.3f}")

mean_all = np.mean([dice_score(pred7, mask7, label) for label in [0, 1, 2]])
mean_fg  = np.mean([dice_score(pred7, mask7, label) for label in [1, 2]])
print(f"\nMean Dice, all three classes:     {mean_all:.3f}")
print(f"Mean Dice, cytoplasm + nucleus:   {mean_fg:.3f}")

Background is the easy class — bright, empty glass — and it pulls the average up without
telling us anything about the cells.  From now on, **"Dice" means the average of the
cytoplasm and nucleus Dice**.

### 2.6 The whole dataset

One image is an anecdote.  Let's wrap the segmenter in a function and score all 200 images.

In [ ]:
def segment(gray, t_nucleus, t_background):
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nucleus] = 2
    pred[(gray >= t_nucleus) & (gray < t_background)] = 1
    return pred

def image_dice(pred, target):
    return (dice_score(pred, target, 1) + dice_score(pred, target, 2)) / 2

def mean_dice(grays, masks, t_nucleus, t_background):
    return np.mean([image_dice(segment(g, t_nucleus, t_background), m) for g, m in zip(grays, masks)])

scores = np.array([image_dice(segment(gray_all[i], t_nucleus, t_background), y[i]) for i in range(200)])
print(f"Mean Dice over all 200 images: {scores.mean():.4f}")

plt.hist(scores, bins=20, range=(0, 1), color='seagreen', edgecolor='white')
plt.xlabel('Dice (cytoplasm + nucleus)')
plt.ylabel('number of images')
plt.title('Dice score of every image')
plt.show()

In [ ]:
worst = np.argmin(scores)
pred_worst = segment(gray_all[worst], t_nucleus, t_background)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(X[worst].transpose(1, 2, 0))
axes[0].set_title(f'Worst image: {worst}   (Dice {scores[worst]:.3f})')
axes[1].imshow(y[worst], cmap='viridis', vmin=0, vmax=2, interpolation='nearest')
axes[1].set_title('Ground truth')
axes[2].imshow(pred_worst, cmap='viridis', vmin=0, vmax=2, interpolation='nearest')
axes[2].set_title('Prediction')
for ax in axes:
    ax.axis('off')
plt.show()

**Think about it:** why does thresholding fail so badly on this image?  Is it the threshold
values, or something that no threshold could fix?

## 3. Noise and Denoising

Every camera adds small random errors to each pixel — **noise**.  Thresholding looks at one
pixel at a time, so a single noisy pixel can flip to the wrong side of a threshold.  Let's
see how much that hurts, and whether we can undo it.

📖 Book: [§2.6.1 Why Noise Matters in Computer Vision](https://obooks.tech/imageprocessing/chapter2.html#why-noise-matters-in-computer-vision)

We switch to **image 0** here — thresholding does well on it, so the damage done by noise is easy to see.

### 🧩 Puzzle 6 — Add Gaussian noise

**Gaussian noise** adds a random number to every pixel, drawn from a bell curve centred
on 0.  Its standard deviation (σ) sets how strong the noise is.  Add noise with σ = 0.2,
then **clip** the result so that it stays between 0 and 1.

📖 Book: [§2.6.2 Types of Noise and Generation Methods](https://obooks.tech/imageprocessing/chapter2.html#types-of-noise-and-generation-methods)

<details><summary><b>💡 Hint 1</b> (click to open)</summary>

`np.random.normal(mean, standard_deviation, size)` draws random numbers from a bell curve.

</details>

<details><summary><b>💡 Hint 2</b> (click to open)</summary>

`np.clip(a, low, high)` pushes anything below `low` up to `low` and anything above `high` down to `high`.

</details>

In [ ]:
gray0 = gray_all[0]
mask0 = y[0]

np.random.seed(0)                      # same "random" noise every time we run
noise  = np.random.normal(___, ___, size=(256, 256))
noisy0 = np.clip(gray0 + noise, ___, ___)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0, 0].imshow(gray0, cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Clean')
axes[0, 1].imshow(noisy0, cmap='gray', vmin=0, vmax=1)
axes[0, 1].set_title('Noisy (σ = 0.2)')
axes[1, 0].imshow(segment(gray0, 0.45, 0.85), cmap='viridis', vmin=0, vmax=2, interpolation='nearest')
axes[1, 0].set_title(f'Clean → Dice {image_dice(segment(gray0, 0.45, 0.85), mask0):.3f}')
axes[1, 1].imshow(segment(noisy0, 0.45, 0.85), cmap='viridis', vmin=0, vmax=2, interpolation='nearest')
axes[1, 1].set_title(f'Noisy → Dice {image_dice(segment(noisy0, 0.45, 0.85), mask0):.3f}')
for ax in axes.flat:
    ax.axis('off')
plt.show()

# ── Check ──
assert abs(noise.mean()) < 0.01 and abs(noise.std() - 0.2) < 0.01, "The noise should have mean 0 and standard deviation 0.2"
assert noisy0.min() >= 0 and noisy0.max() <= 1, "Clip noisy0 to the range 0 to 1"
print("✅ Puzzle 6 solved!")

The prediction is now peppered with specks, and the Dice drops.

### 🧩 Puzzle 7 — Two denoising filters

Both filters replace each pixel using its **neighbours**:

- **Gaussian filter** — a weighted average of nearby pixels.  `sigma` sets how far it
  reaches: bigger `sigma` = smoother, but blurrier edges.
- **Median filter** — the *middle* value of a `size × size` square of neighbours.  A single
  extreme pixel cannot drag the median, so edges stay sharper.

Try `sigma=2` for the Gaussian and `size=5` for the median.  Then experiment: can you beat them?

📖 Book: [§2.6.4 Denoising Methods](https://obooks.tech/imageprocessing/chapter2.html#denoising-methods)

<details><summary><b>💡 Hint 1</b> (click to open)</summary>

Both functions take the image first, then one setting: `gaussian_filter(image, sigma=...)` and `median_filter(image, size=...)`.

</details>

<details><summary><b>💡 Hint 2</b> (click to open)</summary>

Very small settings barely change anything; very large ones blur the nucleus away.  Somewhere between 1 and 7 is sensible.

</details>

In [ ]:
from scipy.ndimage import gaussian_filter, median_filter

gauss0  = gaussian_filter(noisy0, sigma=___)
median0 = median_filter(noisy0, size=___)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for col, (name, img) in enumerate([('Noisy', noisy0), ('Gaussian filter', gauss0), ('Median filter', median0)]):
    pred = segment(img, 0.45, 0.85)
    axes[0, col].imshow(img, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(name)
    axes[1, col].imshow(pred, cmap='viridis', vmin=0, vmax=2, interpolation='nearest')
    axes[1, col].set_title(f'Dice {image_dice(pred, mask0):.3f}')
for ax in axes.flat:
    ax.axis('off')
plt.show()

# ── Check ──
noisy_dice = image_dice(segment(noisy0, 0.45, 0.85), mask0)
assert image_dice(segment(gauss0, 0.45, 0.85), mask0) > noisy_dice + 0.05, "The Gaussian filter should clearly help — check sigma"
assert image_dice(segment(median0, 0.45, 0.85), mask0) > noisy_dice + 0.05, "The median filter should clearly help — check size"
print("✅ Puzzle 7 solved!")

Different noise calls for different filters.  **Salt-and-pepper noise** turns a few random
pixels pure black or pure white, like dust or dead camera pixels.  The table below
compares both filters on both kinds of noise.

📖 Book: [§2.6.5 Comparison of Denoising Methods](https://obooks.tech/imageprocessing/chapter2.html#comparison-of-denoising-methods)

In [ ]:
np.random.seed(0)
salt_pepper0 = gray0.copy()
r = np.random.random((256, 256))
salt_pepper0[r < 0.05] = 0      # 5% of pixels turned black
salt_pepper0[r > 0.95] = 1      # 5% of pixels turned white

print(f"{'':>16} {'no filter':>10} {'Gaussian':>10} {'median':>10}")
for name, img in [('Gaussian noise', noisy0), ('salt-and-pepper', salt_pepper0)]:
    d_none   = image_dice(segment(img, 0.45, 0.85), mask0)
    d_gauss  = image_dice(segment(gaussian_filter(img, sigma=2), 0.45, 0.85), mask0)
    d_median = image_dice(segment(median_filter(img, size=3), 0.45, 0.85), mask0)
    print(f"{name:>16} {d_none:>10.3f} {d_gauss:>10.3f} {d_median:>10.3f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(salt_pepper0, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Salt-and-pepper noise')
axes[1].imshow(gaussian_filter(salt_pepper0, sigma=2), cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Gaussian filter (sigma=2)')
axes[2].imshow(median_filter(salt_pepper0, size=3), cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Median filter (size=3)')
for ax in axes:
    ax.axis('off')
plt.show()

**Think about it:** the Dice scores of the two filters are close, but look at the pictures.
Which filter actually *removed* the specks, and which one only smeared them into grey
smudges?  A small `size=3` median was enough here — why does a median ignore a lone black
pixel, while an average cannot?

### Does denoising help on the *real* images?

Our images were never artificially noised — but real microscope images carry some noise
of their own, and cells have grainy texture.  Let's smooth all 200 images and see whether the
dataset Dice changes.

### 🧩 Puzzle 8 — Smooth the whole stack in one call

`gaussian_filter` works on arrays of any dimension.  `gray_all` has shape `(200, 256, 256)`,
and you can give a *separate* `sigma` for each axis.  We want to smooth along rows and
columns, but **never** blend one image into the next one.

📖 Book: [§2.6.4 Denoising Methods](https://obooks.tech/imageprocessing/chapter2.html#denoising-methods) and
[§1.5 Four-Dimensional Arrays: Batch Processing](https://obooks.tech/imageprocessing/chapter1.html#four-dimensional-arrays-batch-processing)

<details><summary><b>💡 Hint 1</b> (click to open)</summary>

The axes of `gray_all` are (image, row, column).  Along which axis must the filter do nothing?

</details>

<details><summary><b>💡 Hint 2</b> (click to open)</summary>

A `sigma` of 0 along an axis means “no smoothing in that direction”.

</details>

In [ ]:
smooth_all = gaussian_filter(gray_all, sigma=(___, 2, 2))

print(f"Mean Dice, raw grayscale:   {mean_dice(gray_all,   y, 0.45, 0.85):.4f}")
print(f"Mean Dice, smoothed first:  {mean_dice(smooth_all, y, 0.45, 0.85):.4f}")

# ── Check ──
assert np.allclose(smooth_all[5], gaussian_filter(gray_all[5], sigma=2), atol=1e-6), \
    "Each image should be smoothed on its own, without mixing in its neighbours in the stack"
print("✅ Puzzle 8 solved!")

A small but real improvement, for free.  But notice *how* we chose `sigma=2`: by trying
values and keeping the one that looked best **on all 200 images**.  That is a problem, as
the next section explains.

## 4. Honest Evaluation: Train and Test

So far we have picked thresholds and filter settings by looking at how well they did on the
very images we then reported scores for.  That is like studying the exact questions that
will be on the exam: the score is real, but it says little about *new* questions.

The fix is to split the images **before** tuning anything:

- **training set** (80%) — use it as much as you like to choose settings;
- **test set** (20%) — hide it until the very end, then score it **once**.

📖 Book: [Appendix C — Train and Test Split](https://obooks.tech/imageprocessing/appendixc.html#train-and-test-split) and
[Appendix C — Stratification](https://obooks.tech/imageprocessing/appendixc.html#stratification)

A purely random split could, by bad luck, put most of the large-nucleus images on one side.
**Stratifying** prevents that: we sort images into four groups (quartiles) by nucleus size,
and the split keeps the same mix of groups on both sides.

### 🧩 Puzzle 9 — A stratified 80/20 split

<details><summary><b>💡 Hint 1</b> (click to open)</summary>

Quartile boundaries are the 25th, 50th and 75th percentiles.

</details>

<details><summary><b>💡 Hint 2</b> (click to open)</summary>

`test_size` is the *fraction* of images to hold out.

</details>

In [ ]:
from sklearn.model_selection import train_test_split

nuc_px   = (y == 2).sum(axis=(1, 2))
quartile = np.digitize(nuc_px, np.percentile(nuc_px, [___]))

train_idx, test_idx = train_test_split(np.arange(200), test_size=___, stratify=quartile, random_state=42)

print("Training images:", len(train_idx))
print("Test images:    ", len(test_idx))
print("Images per nucleus-size group, train:", np.bincount(quartile[train_idx]))
print("Images per nucleus-size group, test: ", np.bincount(quartile[test_idx]))

# ── Check ──
assert len(train_idx) == 160 and len(test_idx) == 40, "Hold out 20% of the 200 images"
assert set(np.unique(quartile)) == {0, 1, 2, 3}, "You should end up with four groups (0, 1, 2, 3)"
print("✅ Puzzle 9 solved!")

Now we tune `t_nucleus` using the **training images only**.  We try seven candidate values
and record the training Dice of each (we keep `t_background = 0.85` and the `sigma=2`
smoothing from Section 3).

### 🧩 Puzzle 10 — Pick the best threshold

`train_scores[k]` is the training Dice of `candidates[k]`.  Find the *position* of the
highest score, and use it to look up the matching candidate.

📖 Book: [§1.2.4 Aggregation Functions](https://obooks.tech/imageprocessing/chapter1.html#aggregation-functions)

<details><summary><b>💡 Hint</b> (click to open)</summary>

`np.max` gives the highest *value*.  There is a sister function that gives the *position* of the highest value.

</details>

In [ ]:
candidates   = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
train_scores = [mean_dice(smooth_all[train_idx], y[train_idx], t, 0.85) for t in candidates]

best_t = candidates[np.___(train_scores)]
print("Best t_nucleus on the training set:", best_t)

plt.plot(candidates, train_scores, 'o-', color='steelblue')
plt.xlabel('t_nucleus')
plt.ylabel('mean Dice (training images)')
plt.title('Tuning t_nucleus on the training set')
plt.show()

# ── Check ──
assert best_t == candidates[int(np.argmax(train_scores))], "best_t should be the candidate with the highest training Dice"
print("✅ Puzzle 10 solved!")

The moment of truth — score the test images, **once**:

In [ ]:
train_dice = mean_dice(smooth_all[train_idx], y[train_idx], best_t, 0.85)
test_dice  = mean_dice(smooth_all[test_idx],  y[test_idx],  best_t, 0.85)

print(f"Chosen settings: sigma = 2, t_nucleus = {best_t}, t_background = 0.85")
print(f"  Training Dice: {train_dice:.4f}")
print(f"  Test Dice:     {test_dice:.4f}")

**Think about it:**
1. Is the test Dice higher or lower than the training Dice?  Would you expect it to be lower?
   (Hint: a thresholding rule has only two knobs.  See
   [Appendix C — Overfitting and Underfitting](https://obooks.tech/imageprocessing/appendixc.html#overfitting-and-underfitting).)
2. We chose `sigma=2` in Section 3 by looking at **all 200 images** — including the 40 test
   images.  Does that make our test score less trustworthy?  How should we have chosen `sigma`?
   See [Appendix C — Data Leakage](https://obooks.tech/imageprocessing/appendixc.html#data-leakage).
3. If you ran the split again with a different `random_state`, would you get the same test
   Dice?  What does that tell you about reporting a single number?

## 5. Back to the Goal: the N/C Ratio

Dice measures how well the *shapes* overlap.  But the number a pathologist wants is the
**N/C ratio**.  How close does our tuned segmenter get on the test images?

Each dot below is one test image.  Dots on the dashed line are perfect predictions.  We sum
up the fit with **R² against the line y = x**: 1 means every dot is on the line, and it
drops the further the dots stray — whether they scatter, or sit consistently too high or too low.

In [ ]:
def nc_ratio(mask):
    return (mask == 2).sum() / (mask == 1).sum()

nc_true = np.array([nc_ratio(y[i]) for i in test_idx])
nc_pred = np.array([nc_ratio(segment(smooth_all[i], best_t, 0.85)) for i in test_idx])

r2 = 1 - np.sum((nc_pred - nc_true) ** 2) / np.sum((nc_true - nc_true.mean()) ** 2)

plt.figure(figsize=(5, 5))
plt.scatter(nc_true, nc_pred, color='steelblue', alpha=0.7)
plt.plot([0, 7], [0, 7], 'r--')
plt.xlim(0, 7)
plt.ylim(0, 7)
plt.xlabel('true N/C ratio')
plt.ylabel('predicted N/C ratio')
plt.title(f'Test images — R² = {r2:.3f}')
plt.show()

R² can drop **below zero**.  A negative R² means our predictions are *worse than simply
guessing the average N/C ratio for every image*.  Most dots sit near the line, but a few
images are so far off that they swamp the rest — look at the dots at the far right and
top of the plot.  Even a respectable Dice of about 0.72 does not guarantee a usable N/C ratio.

## Wrap-up

**Key takeaways:**
- A threshold turns an image into a `True`/`False` mask; two thresholds give a three-class segmenter.
- **Accuracy misleads when classes are imbalanced.**  Always compare against a lazy baseline.
- **Dice** (= F1) and **IoU** measure overlap one class at a time; we report the average of
  cytoplasm and nucleus Dice.
- **Denoising** before thresholding repairs a lot of noise damage.  Gaussian filters average,
  median filters take the middle value and cope better with isolated extreme pixels.
- **Tune on training images, report on test images — once.**  Any setting chosen by
  looking at test images (even just by eye) makes the test score optimistic.

**The limit of thresholding:** Lab 1's histogram showed cytoplasm and nucleus overlap in
brightness.  No choice of thresholds or filters can separate pixels that have the *same*
brightness.  To do better we need methods that look at colour, texture and shape — that is
where the rest of the course is heading.

---

### Group discussion 👥

1. Find the test image with the **largest** N/C error.  Display it next to its mask and
   prediction.  What went wrong?
2. A pathologist needs the N/C ratio to within ±0.05.  Does our segmenter meet that bar?
   On how many of the 40 test images?
3. Give one example, from this lab, of each: a number that looked good but was misleading;
   a decision that quietly used the test images.